In [ ]:
# CELL 1 — Install dependencies
# Run once, then RESTART KERNEL, then run all from Cell 2
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('STDERR:', r.stderr[-300:])
    else:
        print(f'OK: {cmd[:70]}')

# Downgrade numpy to <2 — torch on Kaggle was compiled against numpy 1.x
run('pip install -q "numpy<2"')
# XGBoost GPU
run('pip install -q --upgrade xgboost')
# LightGBM (already has GPU support)
run('pip install -q --upgrade lightgbm')
# Fix torch
run('pip install -q --force-reinstall torch==2.2.0 --index-url https://download.pytorch.org/whl/cu118')

import numpy as np
import xgboost as xgb
import lightgbm as lgb
import torch
print(f'numpy:    {np.__version__}')
print(f'xgboost:  {xgb.__version__}')
print(f'lightgbm: {lgb.__version__}')
print(f'torch:    {torch.__version__}  cuda: {torch.cuda.is_available()}')
print()
print('*** RESTART KERNEL, then Run All from Cell 2 ***')

In [1]:
# CELL 2 — Imports & config
import os, gc, time, pickle, copy, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import psutil
import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def ram(): return psutil.virtual_memory().used / 1e9

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = '/kaggle/input/datasets/boomboom911/eda-1234'
WORK_DIR = '/kaggle/working'

print(f'numpy:    {np.__version__}')
print(f'torch:    {torch.__version__}  device: {DEVICE}')
print(f'xgboost:  {xgb.__version__}')
print(f'lightgbm: {lgb.__version__}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU RAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

print('\nScanning data files:')
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        p = os.path.join(root, f)
        print(f'  {p}  ({os.path.getsize(p)//1024//1024} MB)')
print('Cell 2 done ✓')

numpy:    2.0.2
torch:    2.10.0+cu128  device: cuda
xgboost:  3.2.0
lightgbm: 4.6.0
GPU: Tesla P100-PCIE-16GB
GPU RAM: 17.1 GB

Scanning data files:
  /kaggle/input/datasets/boomboom911/eda-1234/taxi_zone_lookup.csv  (0 MB)
  /kaggle/input/datasets/boomboom911/eda-1234/taxi_zone_meta.csv  (0 MB)
  /kaggle/input/datasets/boomboom911/eda-1234/sample_1M.parquet  (13 MB)
  /kaggle/input/datasets/boomboom911/eda-1234/train.parquet  (328 MB)
  /kaggle/input/datasets/boomboom911/eda-1234/process_metadata.py  (0 MB)
  /kaggle/input/datasets/boomboom911/eda-1234/dev.parquet  (11 MB)
Cell 2 done ✓


In [2]:
# CELL 3 — Load data
import os, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

_train_path = os.path.join(DATA_DIR, 'train.parquet')
_dev_path   = os.path.join(DATA_DIR, 'dev.parquet')

_avail    = set(pq.read_schema(_train_path).names)
HAS_PAX   = 'passenger_count' in _avail
LOAD_COLS = ['pickup_zone','dropoff_zone','requested_at','duration_seconds'] + (['passenger_count'] if HAS_PAX else [])
print(f'Columns available: {sorted(_avail)}')
print(f'Loading: {LOAD_COLS}')

print(f'RAM before: {ram():.1f}GB')
t0 = time.time()

train = pd.read_parquet(_train_path, columns=LOAD_COLS)
train['pickup_zone']      = train['pickup_zone'].clip(1,265).astype('int16')
train['dropoff_zone']     = train['dropoff_zone'].clip(1,265).astype('int16')
train['duration_seconds'] = train['duration_seconds'].astype('float32')
train['passenger_count']  = train['passenger_count'].clip(1,9).astype('int8') if HAS_PAX else np.int8(1)
print(f'train: {len(train):,} rows  {time.time()-t0:.0f}s  RAM:{ram():.1f}GB')

dev = pd.read_parquet(_dev_path, columns=LOAD_COLS)
dev['pickup_zone']      = dev['pickup_zone'].clip(1,265).astype('int16')
dev['dropoff_zone']     = dev['dropoff_zone'].clip(1,265).astype('int16')
dev['duration_seconds'] = dev['duration_seconds'].astype('float32')
dev['passenger_count']  = dev['passenger_count'].clip(1,9).astype('int8') if HAS_PAX else np.int8(1)
print(f'dev:   {len(dev):,} rows')

zone_meta    = pd.read_csv(os.path.join(DATA_DIR, 'taxi_zone_meta.csv'))
borough_cats = zone_meta.set_index('LocationID')['Borough'].astype('category').cat.codes
print(f'zone_meta: {zone_meta.shape}')
print(f'RAM: {ram():.1f}GB')
print('Cell 3 done ✓')

Columns available: ['dropoff_zone', 'duration_seconds', 'passenger_count', 'pickup_zone', 'requested_at']
Loading: ['pickup_zone', 'dropoff_zone', 'requested_at', 'duration_seconds', 'passenger_count']
RAM before: 1.9GB
train: 36,700,289 rows  18s  RAM:4.7GB
dev:   1,230,911 rows
zone_meta: (265, 6)
RAM: 4.8GB
Cell 3 done ✓


In [3]:
# CELL 4 — Build lookup statistics
import gc, time
import numpy as np
import pandas as pd

print(f'Building stats on {len(train):,} rows...  RAM:{ram():.1f}GB')
t0 = time.time()

ts = pd.to_datetime(train['requested_at'])
hr = ts.dt.hour.astype('int8')
dw = ts.dt.dayofweek.astype('int8')
mo = ts.dt.month.astype('int8')

global_mean   = float(train['duration_seconds'].mean())
global_median = float(train['duration_seconds'].median())
global_p75    = float(train['duration_seconds'].quantile(0.75))

pair_agg = (
    train.groupby(['pickup_zone','dropoff_zone'])['duration_seconds']
    .agg(['mean','median','std','count'])
    .rename(columns={'mean':'pair_mean','median':'pair_median','std':'pair_std','count':'pair_count'})
)
pair_agg['pair_std'] = pair_agg['pair_std'].fillna(0.0)

pu_mean = train.groupby('pickup_zone')['duration_seconds'].mean()
do_mean = train.groupby('dropoff_zone')['duration_seconds'].mean()

hour_pu_mean = train.assign(_h=hr).groupby(['_h','pickup_zone'])['duration_seconds'].mean()
hour_pu_mean.index.names = ['hour','pickup_zone']
hour_do_mean = train.assign(_h=hr).groupby(['_h','dropoff_zone'])['duration_seconds'].mean()
hour_do_mean.index.names = ['hour','dropoff_zone']

pair_hour_mean = train.assign(_h=hr).groupby(['pickup_zone','dropoff_zone','_h'])['duration_seconds'].mean()
pair_hour_mean.index.names = ['pickup_zone','dropoff_zone','hour']
pair_dow_mean  = train.assign(_d=dw).groupby(['pickup_zone','dropoff_zone','_d'])['duration_seconds'].mean()
pair_dow_mean.index.names  = ['pickup_zone','dropoff_zone','dow']
pair_month_mean = train.assign(_m=mo).groupby(['pickup_zone','dropoff_zone','_m'])['duration_seconds'].mean()
pair_month_mean.index.names = ['pickup_zone','dropoff_zone','month']

bc = borough_cats
train_boro_pu = train['pickup_zone'].map(bc).fillna(-1).astype('int8')
train_boro_do = train['dropoff_zone'].map(bc).fillna(-1).astype('int8')
boro_pair_mean = train.assign(_bp=train_boro_pu,_bd=train_boro_do).groupby(['_bp','_bd'])['duration_seconds'].mean()
boro_pair_mean.index.names = ['pu_boro','do_boro']

train_y    = train['duration_seconds'].values.astype('float32')
dev_y      = dev['duration_seconds'].values.astype('float32')
n_train    = len(train)
train_slim = train[['pickup_zone','dropoff_zone','requested_at','passenger_count']].copy()
del train, ts, hr, dw, mo, train_boro_pu, train_boro_do
gc.collect()

stats = dict(
    pair_agg=pair_agg, pu_mean=pu_mean, do_mean=do_mean,
    hour_pu_mean=hour_pu_mean, hour_do_mean=hour_do_mean,
    pair_hour_mean=pair_hour_mean, pair_dow_mean=pair_dow_mean,
    pair_month_mean=pair_month_mean, boro_pair_mean=boro_pair_mean,
    global_mean=global_mean, global_median=global_median, global_p75=global_p75,
    borough_cats=bc, zone_meta=zone_meta,
)
print(f'done {time.time()-t0:.0f}s | pairs:{len(pair_agg):,} pair-hour:{len(pair_hour_mean):,}')
print(f'global_mean={global_mean:.1f}s  RAM:{ram():.1f}GB')
print('Cell 4 done ✓')

Building stats on 36,700,289 rows...  RAM:4.2GB
done 38s | pairs:44,697 pair-hour:407,022
global_mean=989.4s  RAM:4.2GB
Cell 4 done ✓


In [4]:
# CELL 5 — Feature engineering
import gc, time
import numpy as np
import pandas as pd

AIRPORTS = frozenset({1, 132, 138})

_zm      = zone_meta.set_index('LocationID')
_lat_col = next((c for c in _zm.columns if c.lower() in ('lat','latitude','y')), None)
_lon_col = next((c for c in _zm.columns if c.lower() in ('lon','lng','longitude','x')), None)
_HAS_COORDS = bool(_lat_col and _lon_col)
if _HAS_COORDS:
    _zlat = _zm[_lat_col].to_dict()
    _zlon = _zm[_lon_col].to_dict()
    print(f'Centroids: {_lat_col}, {_lon_col}')
else:
    print('No centroids — dist=0')

def _haversine(lat1, lon1, lat2, lon2):
    dlat = np.radians(lat2-lat1); dlon = np.radians(lon2-lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(dlon/2)**2
    a = np.clip(a, 0.0, 1.0)
    return (6371.0 * 2 * np.arcsin(np.sqrt(a))).astype('float32')

def engineer(df, stats):
    df   = df.copy()
    ts   = pd.to_datetime(df['requested_at'])
    pu   = df['pickup_zone'].values.astype('int32')
    do   = df['dropoff_zone'].values.astype('int32')
    pax  = df['passenger_count'].values.astype('int32') if 'passenger_count' in df.columns else np.ones(len(df),'int32')
    h_v  = ts.dt.hour.values.astype('int32')
    dw_v = ts.dt.dayofweek.values.astype('int32')
    mo_v = ts.dt.month.values.astype('int32')
    dy_v = ts.dt.day.values.astype('int32')
    wk_v = ts.dt.isocalendar().week.values.astype('int32')

    bc  = stats['borough_cats']
    pub = np.clip(np.array([int(bc.get(z,-1))+1 for z in pu],'int32'), 0, 7)
    dob = np.clip(np.array([int(bc.get(z,-1))+1 for z in do],'int32'), 0, 7)
    gm  = stats['global_mean']; gmd = stats['global_median']

    tmp = pd.DataFrame({'pickup_zone':pu,'dropoff_zone':do,'hour':h_v,'dow':dw_v,'month':mo_v,'pu_boro':pub,'do_boro':dob})

    pa = stats['pair_agg'].reset_index()
    tmp = tmp.merge(pa[['pickup_zone','dropoff_zone','pair_mean','pair_median','pair_std','pair_count']], on=['pickup_zone','dropoff_zone'], how='left')
    tmp['pair_mean']   = tmp['pair_mean'].fillna(gm)
    tmp['pair_median'] = tmp['pair_median'].fillna(gmd)
    tmp['pair_std']    = tmp['pair_std'].fillna(stats['global_p75'])
    tmp['pair_count']  = tmp['pair_count'].fillna(0.0)

    ph = stats['pair_hour_mean'].reset_index(); ph.columns=['pickup_zone','dropoff_zone','hour','pair_hour_mean']
    tmp = tmp.merge(ph, on=['pickup_zone','dropoff_zone','hour'], how='left')
    tmp['pair_hour_mean'] = tmp['pair_hour_mean'].fillna(tmp['pair_mean'])

    pd_ = stats['pair_dow_mean'].reset_index(); pd_.columns=['pickup_zone','dropoff_zone','dow','pair_dow_mean']
    tmp = tmp.merge(pd_, on=['pickup_zone','dropoff_zone','dow'], how='left')
    tmp['pair_dow_mean'] = tmp['pair_dow_mean'].fillna(tmp['pair_mean'])

    pm_ = stats['pair_month_mean'].reset_index(); pm_.columns=['pickup_zone','dropoff_zone','month','pair_month_mean']
    tmp = tmp.merge(pm_, on=['pickup_zone','dropoff_zone','month'], how='left')
    tmp['pair_month_mean'] = tmp['pair_month_mean'].fillna(tmp['pair_mean'])

    tmp = tmp.merge(stats['pu_mean'].rename('pu_zone_mean').reset_index(), on='pickup_zone', how='left')
    tmp['pu_zone_mean'] = tmp['pu_zone_mean'].fillna(gm)
    tmp = tmp.merge(stats['do_mean'].rename('do_zone_mean').reset_index(), on='dropoff_zone', how='left')
    tmp['do_zone_mean'] = tmp['do_zone_mean'].fillna(gm)

    hpu = stats['hour_pu_mean'].reset_index(); hpu.columns=['hour','pickup_zone','hour_pu_mean']
    tmp = tmp.merge(hpu, on=['hour','pickup_zone'], how='left')
    tmp['hour_pu_mean'] = tmp['hour_pu_mean'].fillna(tmp['pu_zone_mean'])

    hdo = stats['hour_do_mean'].reset_index(); hdo.columns=['hour','dropoff_zone','hour_do_mean']
    tmp = tmp.merge(hdo, on=['hour','dropoff_zone'], how='left')
    tmp['hour_do_mean'] = tmp['hour_do_mean'].fillna(tmp['do_zone_mean'])

    bpm = stats['boro_pair_mean'].reset_index(); bpm.columns=['pu_boro','do_boro','boro_pair_mean']
    tmp = tmp.merge(bpm, on=['pu_boro','do_boro'], how='left')
    tmp['boro_pair_mean'] = tmp['boro_pair_mean'].fillna(gm)

    if _HAS_COORDS:
        D, L = 40.7128, -74.006
        pu_lat = np.array([_zlat.get(z,D) for z in pu],'float32')
        pu_lon = np.array([_zlon.get(z,L) for z in pu],'float32')
        do_lat = np.array([_zlat.get(z,D) for z in do],'float32')
        do_lon = np.array([_zlon.get(z,L) for z in do],'float32')
        dist      = _haversine(pu_lat, pu_lon, do_lat, do_lon)
        manhattan = ((np.abs(do_lat-pu_lat) + np.abs(do_lon-pu_lon)) * 111.0).astype('float32')
    else:
        dist = manhattan = np.zeros(len(pu),'float32')

    is_christmas = ((mo_v==12) & (dy_v>=23) & (dy_v<=26)).astype('float32')
    is_nye       = ((mo_v==12) & (dy_v>=30)).astype('float32')

    X = np.column_stack([
        tmp['pair_mean'].values.astype('float32'),
        tmp['pair_median'].values.astype('float32'),
        tmp['pair_std'].values.astype('float32'),
        np.log1p(tmp['pair_count'].values).astype('float32'),
        tmp['pair_hour_mean'].values.astype('float32'),
        tmp['pair_dow_mean'].values.astype('float32'),
        tmp['pair_month_mean'].values.astype('float32'),
        tmp['pu_zone_mean'].values.astype('float32'),
        tmp['do_zone_mean'].values.astype('float32'),
        tmp['hour_pu_mean'].values.astype('float32'),
        tmp['hour_do_mean'].values.astype('float32'),
        tmp['boro_pair_mean'].values.astype('float32'),
        dist, manhattan,
        h_v.astype('float32'), dw_v.astype('float32'), mo_v.astype('float32'),
        dy_v.astype('float32'), wk_v.astype('float32'),
        np.sin(2*np.pi*h_v/24).astype('float32'),
        np.cos(2*np.pi*h_v/24).astype('float32'),
        (dw_v/6.0).astype('float32'),
        pu.astype('float32'), do.astype('float32'),
        pub.astype('float32'), dob.astype('float32'),
        (pu==do).astype('float32'), (pub==dob).astype('float32'),
        np.array([float(z in AIRPORTS) for z in pu],'float32'),
        np.array([float(z in AIRPORTS) for z in do],'float32'),
        np.maximum(is_christmas, is_nye), is_christmas, is_nye,
        pax.astype('float32'),
    ]).astype('float32')

    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)


FEATURE_NAMES = [
    'pair_mean','pair_median','pair_std','log_pair_count',
    'pair_hour_mean','pair_dow_mean','pair_month_mean',
    'pu_zone_mean','do_zone_mean','hour_pu_mean','hour_do_mean','boro_pair_mean',
    'dist_km','manhattan_km',
    'hour','dow','month','day','week',
    'hour_sin','hour_cos','dow_norm',
    'pickup_zone','dropoff_zone','pu_boro','do_boro',
    'same_zone','same_boro','is_airport_pu','is_airport_do',
    'is_holiday','is_christmas','is_nye','pax',
]
assert len(FEATURE_NAMES) == 34, f'Expected 34 features, got {len(FEATURE_NAMES)}'

print('Engineering train features...')
t0 = time.time()
X_train = engineer(train_slim, stats)
print(f'  X_train: {X_train.shape}  NaN={np.isnan(X_train).sum()}  {time.time()-t0:.0f}s  RAM:{ram():.1f}GB')
assert np.isnan(X_train).sum() == 0

print('Engineering dev features...')
X_dev = engineer(dev, stats)
print(f'  X_dev: {X_dev.shape}  NaN={np.isnan(X_dev).sum()}  RAM:{ram():.1f}GB')
assert np.isnan(X_dev).sum() == 0

del train_slim; gc.collect()
print('Cell 5 done ✓')

Centroids: lat, lon
Engineering train features...
  X_train: (36700289, 34)  NaN=0  483s  RAM:9.8GB
Engineering dev features...
  X_dev: (1230911, 34)  NaN=0  RAM:10.1GB
Cell 5 done ✓


In [5]:
# CELL 6 — GPU XGBoost
import time, os
import numpy as np
import xgboost as xgb

print('Training XGBoost on GPU...')
t0 = time.time()

dtrain = xgb.DMatrix(X_train, label=train_y, feature_names=FEATURE_NAMES)
ddev   = xgb.DMatrix(X_dev,   label=dev_y,   feature_names=FEATURE_NAMES)

xgb_params = {
    'objective':        'reg:absoluteerror',
    'tree_method':      'hist',
    'device':           'cuda',
    'max_depth':        6,
    'learning_rate':    0.05,
    'subsample':        0.8,
    'colsample_bytree': 0.7,
    'min_child_weight': 50,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'seed':             42,
    'verbosity':        0,
}

# No early stopping — run fixed 1000 rounds, GPU handles this in ~15 min
xgb_model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=1000,
    evals=[(ddev, 'dev')],
    verbose_eval=100,
)

xgb_preds_dev = xgb_model.predict(ddev).astype('float32')
xgb_mae = float(np.mean(np.abs(xgb_preds_dev - dev_y)))
print(f'\nXGBoost dev MAE: {xgb_mae:.1f}s  {(time.time()-t0)/60:.1f}min')

xgb_model.save_model(os.path.join(WORK_DIR, 'xgb_model.ubj'))
print('Cell 6 done ✓')

Training XGBoost on GPU...
[0]	dev-mae:527.58015
[100]	dev-mae:261.89782
[200]	dev-mae:260.66799
[300]	dev-mae:260.66945
[400]	dev-mae:260.53909
[500]	dev-mae:260.55788
[600]	dev-mae:260.70129
[700]	dev-mae:260.79968
[800]	dev-mae:261.36017
[900]	dev-mae:261.90817
[999]	dev-mae:262.15263

XGBoost dev MAE: 262.2s  4.6min
Cell 6 done ✓


In [9]:
# CELL 7 — GPU LightGBM
import time, os
import numpy as np
import lightgbm as lgb

print('Training LightGBM on GPU...')
t0 = time.time()

lgb_params = {
    'objective':         'regression_l1',
    'device':            'gpu',
    'num_leaves':        255,
    'learning_rate':     0.05,
    'feature_fraction':  0.7,
    'bagging_fraction':  0.8,
    'bagging_freq':      1,
    'min_child_samples': 50,
    'reg_alpha':         0.05,
    'reg_lambda':        0.5,
    'seed':              42,
    'verbosity':         -1,
    'n_jobs':            -1,
}

lgb_train = lgb.Dataset(X_train, label=train_y, feature_name=FEATURE_NAMES, free_raw_data=False)
lgb_dev   = lgb.Dataset(X_dev,   label=dev_y,   feature_name=FEATURE_NAMES, free_raw_data=False, reference=lgb_train)

# No early stopping — run fixed 1000 rounds, GPU ~5 min
lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_dev],
    callbacks=[lgb.log_evaluation(200)],
)

lgb_preds_dev = lgb_model.predict(X_dev).astype('float32')
lgb_mae = float(np.mean(np.abs(lgb_preds_dev - dev_y)))
print(f'\nLightGBM GPU dev MAE: {lgb_mae:.1f}s  best_iteration={lgb_model.best_iteration}  {(time.time()-t0)/60:.1f}min')

cat_preds_dev = lgb_preds_dev  # alias for blend cell
cat_mae = lgb_mae

lgb_model.save_model(os.path.join(WORK_DIR, 'lgb_model.txt'))
print('Cell 7 done ✓')

Training LightGBM on GPU...
[200]	valid_0's l1: 261.998
[400]	valid_0's l1: 263.301
[600]	valid_0's l1: 263.886
[800]	valid_0's l1: 264.121
[1000]	valid_0's l1: 264.353

LightGBM GPU dev MAE: 264.4s  best_iteration=0  34.4min
Cell 7 done ✓


In [3]:
# CELL 7b — Recovery: reload models + rebuild features (run if X_train/xgb_model lost after kernel issues)
# Does NOT retrain anything — just reloads saved files and re-engineers features from parquet
import os, gc, time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import psutil
import xgboost as xgb
import lightgbm as lgb
import pyarrow.parquet as pq

def ram(): return psutil.virtual_memory().used / 1e9

DEVICE   = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
DATA_DIR = '/kaggle/input/datasets/boomboom911/eda-1234'
WORK_DIR = '/kaggle/working'

FEATURE_NAMES = [
    'pair_mean','pair_median','pair_std','log_pair_count',
    'pair_hour_mean','pair_dow_mean','pair_month_mean',
    'pu_zone_mean','do_zone_mean','hour_pu_mean','hour_do_mean','boro_pair_mean',
    'dist_km','manhattan_km',
    'hour','dow','month','day','week',
    'hour_sin','hour_cos','dow_norm',
    'pickup_zone','dropoff_zone','pu_boro','do_boro',
    'same_zone','same_boro','is_airport_pu','is_airport_do',
    'is_holiday','is_christmas','is_nye','pax',
]
assert len(FEATURE_NAMES) == 34

# --- Reload saved models ---
print('Reloading XGBoost...')
xgb_model = xgb.Booster()
xgb_model.load_model(os.path.join(WORK_DIR, 'xgb_model.ubj'))
print('Reloading LightGBM...')
lgb_model = lgb.Booster(model_file=os.path.join(WORK_DIR, 'lgb_model.txt'))
print('Models reloaded ✓')

# --- Reload data + rebuild stats + engineer features (mirrors Cells 3-5) ---
_avail  = set(pq.read_schema(os.path.join(DATA_DIR, 'train.parquet')).names)
HAS_PAX = 'passenger_count' in _avail
LOAD_COLS = ['pickup_zone','dropoff_zone','requested_at','duration_seconds'] + (['passenger_count'] if HAS_PAX else [])

print(f'Loading train...  RAM:{ram():.1f}GB')
train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'), columns=LOAD_COLS)
train['pickup_zone']      = train['pickup_zone'].clip(1,265).astype('int16')
train['dropoff_zone']     = train['dropoff_zone'].clip(1,265).astype('int16')
train['duration_seconds'] = train['duration_seconds'].astype('float32')
train['passenger_count']  = train['passenger_count'].clip(1,9).astype('int8') if HAS_PAX else np.int8(1)

dev = pd.read_parquet(os.path.join(DATA_DIR, 'dev.parquet'), columns=LOAD_COLS)
dev['pickup_zone']      = dev['pickup_zone'].clip(1,265).astype('int16')
dev['dropoff_zone']     = dev['dropoff_zone'].clip(1,265).astype('int16')
dev['duration_seconds'] = dev['duration_seconds'].astype('float32')
dev['passenger_count']  = dev['passenger_count'].clip(1,9).astype('int8') if HAS_PAX else np.int8(1)

zone_meta    = pd.read_csv(os.path.join(DATA_DIR, 'taxi_zone_meta.csv'))
borough_cats = zone_meta.set_index('LocationID')['Borough'].astype('category').cat.codes
print(f'Data loaded  RAM:{ram():.1f}GB')

# Build stats
ts = pd.to_datetime(train['requested_at'])
hr = ts.dt.hour.astype('int8'); dw = ts.dt.dayofweek.astype('int8'); mo = ts.dt.month.astype('int8')
global_mean   = float(train['duration_seconds'].mean())
global_median = float(train['duration_seconds'].median())
global_p75    = float(train['duration_seconds'].quantile(0.75))
pair_agg = train.groupby(['pickup_zone','dropoff_zone'])['duration_seconds'].agg(['mean','median','std','count']).rename(columns={'mean':'pair_mean','median':'pair_median','std':'pair_std','count':'pair_count'})
pair_agg['pair_std'] = pair_agg['pair_std'].fillna(0.0)
pu_mean = train.groupby('pickup_zone')['duration_seconds'].mean()
do_mean = train.groupby('dropoff_zone')['duration_seconds'].mean()
hour_pu_mean = train.assign(_h=hr).groupby(['_h','pickup_zone'])['duration_seconds'].mean(); hour_pu_mean.index.names=['hour','pickup_zone']
hour_do_mean = train.assign(_h=hr).groupby(['_h','dropoff_zone'])['duration_seconds'].mean(); hour_do_mean.index.names=['hour','dropoff_zone']
pair_hour_mean = train.assign(_h=hr).groupby(['pickup_zone','dropoff_zone','_h'])['duration_seconds'].mean(); pair_hour_mean.index.names=['pickup_zone','dropoff_zone','hour']
pair_dow_mean  = train.assign(_d=dw).groupby(['pickup_zone','dropoff_zone','_d'])['duration_seconds'].mean(); pair_dow_mean.index.names=['pickup_zone','dropoff_zone','dow']
pair_month_mean = train.assign(_m=mo).groupby(['pickup_zone','dropoff_zone','_m'])['duration_seconds'].mean(); pair_month_mean.index.names=['pickup_zone','dropoff_zone','month']
bc = borough_cats
boro_pair_mean = train.assign(_bp=train['pickup_zone'].map(bc).fillna(-1).astype('int8'), _bd=train['dropoff_zone'].map(bc).fillna(-1).astype('int8')).groupby(['_bp','_bd'])['duration_seconds'].mean(); boro_pair_mean.index.names=['pu_boro','do_boro']
train_y = train['duration_seconds'].values.astype('float32')
dev_y   = dev['duration_seconds'].values.astype('float32')
train_slim = train[['pickup_zone','dropoff_zone','requested_at','passenger_count']].copy()
del train, ts, hr, dw, mo; gc.collect()
stats = dict(pair_agg=pair_agg, pu_mean=pu_mean, do_mean=do_mean, hour_pu_mean=hour_pu_mean,
             hour_do_mean=hour_do_mean, pair_hour_mean=pair_hour_mean, pair_dow_mean=pair_dow_mean,
             pair_month_mean=pair_month_mean, boro_pair_mean=boro_pair_mean,
             global_mean=global_mean, global_median=global_median, global_p75=global_p75,
             borough_cats=bc, zone_meta=zone_meta)
print(f'Stats built  RAM:{ram():.1f}GB')

# Engineer features (copy of Cell 5 engineer())
AIRPORTS = frozenset({1, 132, 138})
_zm = zone_meta.set_index('LocationID')
_lat_col = next((c for c in _zm.columns if c.lower() in ('lat','latitude','y')), None)
_lon_col = next((c for c in _zm.columns if c.lower() in ('lon','lng','longitude','x')), None)
_HAS_COORDS = bool(_lat_col and _lon_col)
_zlat = _zm[_lat_col].to_dict() if _HAS_COORDS else {}
_zlon = _zm[_lon_col].to_dict() if _HAS_COORDS else {}

def _haversine(lat1, lon1, lat2, lon2):
    dlat = np.radians(lat2-lat1); dlon = np.radians(lon2-lon1)
    a = np.clip(np.sin(dlat/2)**2 + np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(dlon/2)**2, 0.0, 1.0)
    return (6371.0 * 2 * np.arcsin(np.sqrt(a))).astype('float32')

def engineer(df, stats):
    df = df.copy(); ts = pd.to_datetime(df['requested_at'])
    pu = df['pickup_zone'].values.astype('int32'); do = df['dropoff_zone'].values.astype('int32')
    pax = df['passenger_count'].values.astype('int32') if 'passenger_count' in df.columns else np.ones(len(df),'int32')
    h_v = ts.dt.hour.values.astype('int32'); dw_v = ts.dt.dayofweek.values.astype('int32')
    mo_v = ts.dt.month.values.astype('int32'); dy_v = ts.dt.day.values.astype('int32'); wk_v = ts.dt.isocalendar().week.values.astype('int32')
    bc = stats['borough_cats']
    pub = np.clip(np.array([int(bc.get(z,-1))+1 for z in pu],'int32'), 0, 7)
    dob = np.clip(np.array([int(bc.get(z,-1))+1 for z in do],'int32'), 0, 7)
    gm = stats['global_mean']; gmd = stats['global_median']
    tmp = pd.DataFrame({'pickup_zone':pu,'dropoff_zone':do,'hour':h_v,'dow':dw_v,'month':mo_v,'pu_boro':pub,'do_boro':dob})
    pa = stats['pair_agg'].reset_index()
    tmp = tmp.merge(pa[['pickup_zone','dropoff_zone','pair_mean','pair_median','pair_std','pair_count']], on=['pickup_zone','dropoff_zone'], how='left')
    tmp['pair_mean'] = tmp['pair_mean'].fillna(gm); tmp['pair_median'] = tmp['pair_median'].fillna(gmd)
    tmp['pair_std'] = tmp['pair_std'].fillna(stats['global_p75']); tmp['pair_count'] = tmp['pair_count'].fillna(0.0)
    for col, key, idx in [('pair_hour_mean','pair_hour_mean',['pickup_zone','dropoff_zone','hour']),
                           ('pair_dow_mean','pair_dow_mean',['pickup_zone','dropoff_zone','dow']),
                           ('pair_month_mean','pair_month_mean',['pickup_zone','dropoff_zone','month'])]:
        s = stats[key].reset_index(); s.columns = idx + [col]
        tmp = tmp.merge(s, on=idx, how='left'); tmp[col] = tmp[col].fillna(tmp['pair_mean'])
    tmp = tmp.merge(stats['pu_mean'].rename('pu_zone_mean').reset_index(), on='pickup_zone', how='left'); tmp['pu_zone_mean'] = tmp['pu_zone_mean'].fillna(gm)
    tmp = tmp.merge(stats['do_mean'].rename('do_zone_mean').reset_index(), on='dropoff_zone', how='left'); tmp['do_zone_mean'] = tmp['do_zone_mean'].fillna(gm)
    hpu = stats['hour_pu_mean'].reset_index(); hpu.columns=['hour','pickup_zone','hour_pu_mean']
    tmp = tmp.merge(hpu, on=['hour','pickup_zone'], how='left'); tmp['hour_pu_mean'] = tmp['hour_pu_mean'].fillna(tmp['pu_zone_mean'])
    hdo = stats['hour_do_mean'].reset_index(); hdo.columns=['hour','dropoff_zone','hour_do_mean']
    tmp = tmp.merge(hdo, on=['hour','dropoff_zone'], how='left'); tmp['hour_do_mean'] = tmp['hour_do_mean'].fillna(tmp['do_zone_mean'])
    bpm = stats['boro_pair_mean'].reset_index(); bpm.columns=['pu_boro','do_boro','boro_pair_mean']
    tmp = tmp.merge(bpm, on=['pu_boro','do_boro'], how='left'); tmp['boro_pair_mean'] = tmp['boro_pair_mean'].fillna(gm)
    if _HAS_COORDS:
        D,L = 40.7128,-74.006
        pu_lat=np.array([_zlat.get(z,D) for z in pu],'float32'); pu_lon=np.array([_zlon.get(z,L) for z in pu],'float32')
        do_lat=np.array([_zlat.get(z,D) for z in do],'float32'); do_lon=np.array([_zlon.get(z,L) for z in do],'float32')
        dist=_haversine(pu_lat,pu_lon,do_lat,do_lon); manhattan=((np.abs(do_lat-pu_lat)+np.abs(do_lon-pu_lon))*111.0).astype('float32')
    else:
        dist=manhattan=np.zeros(len(pu),'float32')
    is_christmas=((mo_v==12)&(dy_v>=23)&(dy_v<=26)).astype('float32'); is_nye=((mo_v==12)&(dy_v>=30)).astype('float32')
    X = np.column_stack([tmp['pair_mean'].values.astype('float32'),tmp['pair_median'].values.astype('float32'),tmp['pair_std'].values.astype('float32'),np.log1p(tmp['pair_count'].values).astype('float32'),
        tmp['pair_hour_mean'].values.astype('float32'),tmp['pair_dow_mean'].values.astype('float32'),tmp['pair_month_mean'].values.astype('float32'),
        tmp['pu_zone_mean'].values.astype('float32'),tmp['do_zone_mean'].values.astype('float32'),tmp['hour_pu_mean'].values.astype('float32'),tmp['hour_do_mean'].values.astype('float32'),tmp['boro_pair_mean'].values.astype('float32'),
        dist,manhattan,h_v.astype('float32'),dw_v.astype('float32'),mo_v.astype('float32'),dy_v.astype('float32'),wk_v.astype('float32'),
        np.sin(2*np.pi*h_v/24).astype('float32'),np.cos(2*np.pi*h_v/24).astype('float32'),(dw_v/6.0).astype('float32'),
        pu.astype('float32'),do.astype('float32'),pub.astype('float32'),dob.astype('float32'),
        (pu==do).astype('float32'),(pub==dob).astype('float32'),
        np.array([float(z in AIRPORTS) for z in pu],'float32'),np.array([float(z in AIRPORTS) for z in do],'float32'),
        np.maximum(is_christmas,is_nye),is_christmas,is_nye,pax.astype('float32'),
    ]).astype('float32')
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print('Engineering train features...')
X_train = engineer(train_slim, stats)
print(f'  X_train: {X_train.shape}  NaN={np.isnan(X_train).sum()}  RAM:{ram():.1f}GB')
print('Engineering dev features...')
X_dev = engineer(dev, stats)
print(f'  X_dev: {X_dev.shape}  NaN={np.isnan(X_dev).sum()}  RAM:{ram():.1f}GB')
del train_slim; gc.collect()

# Rebuild model predictions for blend
xgb_preds_dev = xgb_model.predict(xgb.DMatrix(X_dev, feature_names=FEATURE_NAMES)).astype('float32')
lgb_preds_dev = lgb_model.predict(X_dev).astype('float32')
cat_preds_dev = lgb_preds_dev
print(f'XGBoost dev MAE: {np.mean(np.abs(xgb_preds_dev - dev_y)):.1f}s')
print(f'LightGBM dev MAE: {np.mean(np.abs(lgb_preds_dev - dev_y)):.1f}s')
print('Cell 7b done ✓ — ready for Cell 8')

Reloading XGBoost...
Reloading LightGBM...
Models reloaded ✓
Loading train...  RAM:5.0GB
Data loaded  RAM:5.3GB
Stats built  RAM:5.3GB
Engineering train features...
  X_train: (36700289, 34)  NaN=0  RAM:11.1GB
Engineering dev features...
  X_dev: (1230911, 34)  NaN=0  RAM:11.3GB
XGBoost dev MAE: 262.2s
LightGBM dev MAE: 264.4s
Cell 7b done ✓ — ready for Cell 8


In [3]:
# CELL 8 — Lightweight NN (residual corrector)
import time, os, copy
import numpy as np
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

FEATURE_NAMES = [
    'pair_mean','pair_median','pair_std','log_pair_count',
    'pair_hour_mean','pair_dow_mean','pair_month_mean',
    'pu_zone_mean','do_zone_mean','hour_pu_mean','hour_do_mean','boro_pair_mean',
    'dist_km','manhattan_km',
    'hour','dow','month','day','week',
    'hour_sin','hour_cos','dow_norm',
    'pickup_zone','dropoff_zone','pu_boro','do_boro',
    'same_zone','same_boro','is_airport_pu','is_airport_do',
    'is_holiday','is_christmas','is_nye','pax',
]
assert len(FEATURE_NAMES) == 34

N_CONT  = 34
N_ZONES = 266

class ETANet(nn.Module):
    def __init__(self):
        super().__init__()
        self.pu_emb   = nn.Embedding(N_ZONES, 16, padding_idx=0)
        self.do_emb   = nn.Embedding(N_ZONES, 16, padding_idx=0)
        self.boro_emb = nn.Embedding(8, 4)
        self.hour_emb = nn.Embedding(24, 6)
        self.dow_emb  = nn.Embedding(7, 4)
        in_dim = 16+16+4+4+6+4 + N_CONT + 1  # 85
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 128),    nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64),     nn.GELU(),
            nn.Linear(64, 1),
        )
    def forward(self, cont, pu, do, pub, dob, h, dow, xgb_pred):
        x = torch.cat([
            self.pu_emb(pu), self.do_emb(do),
            self.boro_emb(pub), self.boro_emb(dob),
            self.hour_emb(h), self.dow_emb(dow),
            cont, xgb_pred.unsqueeze(1),
        ], dim=1)
        return self.net(x).squeeze(1)


class TripDataset(Dataset):
    def __init__(self, X, xgb_pred, pu, do, pub, dob, h, dow, y):
        self.X   = X.astype('float32')
        self.xp  = xgb_pred.astype('float32')
        self.pu  = pu.astype('int32');  self.do  = do.astype('int32')
        self.pub = pub.astype('int32'); self.dob = dob.astype('int32')
        self.h   = h.astype('int32');   self.dow = dow.astype('int32')
        self.y   = y.astype('float32')
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return (
            torch.from_numpy(self.X[i]),
            torch.tensor(int(self.pu[i]),  dtype=torch.long),
            torch.tensor(int(self.do[i]),  dtype=torch.long),
            torch.tensor(int(self.pub[i]), dtype=torch.long),
            torch.tensor(int(self.dob[i]), dtype=torch.long),
            torch.tensor(int(self.h[i]),   dtype=torch.long),
            torch.tensor(int(self.dow[i]), dtype=torch.long),
            torch.tensor(float(self.xp[i]), dtype=torch.float32),
            torch.tensor(float(self.y[i]),  dtype=torch.float32),
        )


IDX    = {n:i for i,n in enumerate(FEATURE_NAMES)}
tr_pu  = X_train[:, IDX['pickup_zone']].astype('int32').clip(1, 265)
tr_do  = X_train[:, IDX['dropoff_zone']].astype('int32').clip(1, 265)
tr_pub = X_train[:, IDX['pu_boro']].astype('int32').clip(0, 7)
tr_dob = X_train[:, IDX['do_boro']].astype('int32').clip(0, 7)
tr_h   = X_train[:, IDX['hour']].astype('int32').clip(0, 23)
tr_dow = X_train[:, IDX['dow']].astype('int32').clip(0, 6)

dv_pu  = X_dev[:, IDX['pickup_zone']].astype('int32').clip(1, 265)
dv_do  = X_dev[:, IDX['dropoff_zone']].astype('int32').clip(1, 265)
dv_pub = X_dev[:, IDX['pu_boro']].astype('int32').clip(0, 7)
dv_dob = X_dev[:, IDX['do_boro']].astype('int32').clip(0, 7)
dv_h   = X_dev[:, IDX['hour']].astype('int32').clip(0, 23)
dv_dow = X_dev[:, IDX['dow']].astype('int32').clip(0, 6)

xgb_preds_train = xgb_model.predict(xgb.DMatrix(X_train, feature_names=FEATURE_NAMES)).astype('float32')

BATCH     = 16384
EPOCHS_NN = 8
LR_NN     = 3e-4
PATIENCE  = 3
CKPT      = os.path.join(WORK_DIR, 'nn_checkpoint.pt')

train_ds = TripDataset(X_train, xgb_preds_train, tr_pu, tr_do, tr_pub, tr_dob, tr_h, tr_dow, train_y)
dev_ds   = TripDataset(X_dev,   xgb_preds_dev,   dv_pu, dv_do, dv_pub, dv_dob, dv_h, dv_dow, dev_y)
train_loader = DataLoader(train_ds, batch_size=BATCH,   shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH*2, shuffle=False, num_workers=2, pin_memory=True)

nn_model  = ETANet().to(DEVICE)
optimizer = torch.optim.AdamW(nn_model.parameters(), lr=LR_NN, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR_NN,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS_NN, pct_start=0.1,
)
criterion = nn.L1Loss()

print(f'ETANet params: {sum(p.numel() for p in nn_model.parameters()):,}')
print(f'Batches/epoch: {len(train_loader):,}  device: {DEVICE}')

best_nn_mae = float('inf'); best_nn_state = None; no_improve = 0

def run_epoch(loader, training):
    nn_model.train(training)
    total, n = 0.0, 0
    t_start = time.time()
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for i, batch in enumerate(loader):
            cb_, pb, db, pub_b, dob_b, hb, dwb, xb, yb = [x.to(DEVICE) for x in batch]
            if training: optimizer.zero_grad()
            pred = nn_model(cb_, pb, db, pub_b, dob_b, hb, dwb, xb)
            loss = criterion(pred, yb)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(nn_model.parameters(), 1.0)
                optimizer.step(); scheduler.step()
            total += loss.item()*len(yb); n += len(yb)
            if training and (i+1) % 100 == 0:
                elapsed  = time.time() - t_start
                rate     = (i+1) / elapsed          # batches/sec
                remaining = (len(loader) - (i+1)) / rate  # seconds left
                print(f'  batch {i+1}/{len(loader)}  mae={total/n:.1f}s  '
                      f'{rate:.1f} b/s  ETA {remaining/60:.1f}min', flush=True)
    return total/max(n,1)

print('\nTraining NN...')
for epoch in range(EPOCHS_NN):
    t0     = time.time()
    tr_mae = run_epoch(train_loader, True)
    dv_mae = run_epoch(dev_loader, False)
    best   = dv_mae < best_nn_mae
    if best:
        best_nn_mae   = dv_mae
        best_nn_state = copy.deepcopy(nn_model.state_dict())
        no_improve    = 0
        torch.save(best_nn_state, CKPT)
    else:
        no_improve += 1
    print(f'Epoch {epoch+1}/{EPOCHS_NN}  tr={tr_mae:.1f}s  dv={dv_mae:.1f}s  '
          f'{(time.time()-t0)/60:.1f}min  lr={scheduler.get_last_lr()[0]:.2e}  '
          f'{"*** BEST" if best else f"(no improve {no_improve}/{PATIENCE})"}')
    if no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1}'); break

nn_model.load_state_dict(best_nn_state)
print(f'\nBest NN dev MAE: {best_nn_mae:.1f}s')

nn_model.eval()
nn_preds_dev = []
with torch.no_grad():
    for batch in dev_loader:
        cb_, pb, db, pub_b, dob_b, hb, dwb, xb, _ = [x.to(DEVICE) for x in batch]
        nn_preds_dev.append(nn_model(cb_, pb, db, pub_b, dob_b, hb, dwb, xb).cpu().numpy())
nn_preds_dev = np.concatenate(nn_preds_dev)
print(f'NN dev MAE: {np.mean(np.abs(nn_preds_dev - dev_y)):.1f}s')
print('Cell 8 done ✓')

ETANet params: 72,717
Batches/epoch: 2,240  device: cuda

Training NN...
  batch 100/2240  mae=989.5s  1.4 b/s  ETA 25.7min
  batch 200/2240  mae=989.4s  1.4 b/s  ETA 23.6min
  batch 300/2240  mae=989.0s  1.5 b/s  ETA 22.2min
  batch 400/2240  mae=988.4s  1.5 b/s  ETA 20.9min
  batch 500/2240  mae=988.0s  1.5 b/s  ETA 19.8min
  batch 600/2240  mae=987.2s  1.5 b/s  ETA 18.6min
  batch 700/2240  mae=986.2s  1.5 b/s  ETA 17.4min
  batch 800/2240  mae=985.0s  1.5 b/s  ETA 16.4min
  batch 900/2240  mae=983.5s  1.5 b/s  ETA 15.2min
  batch 1000/2240  mae=981.4s  1.5 b/s  ETA 14.0min
  batch 1100/2240  mae=978.8s  1.5 b/s  ETA 12.9min
  batch 1200/2240  mae=975.5s  1.5 b/s  ETA 11.8min
  batch 1300/2240  mae=971.1s  1.5 b/s  ETA 10.6min
  batch 1400/2240  mae=965.6s  1.5 b/s  ETA 9.5min
  batch 1500/2240  mae=958.7s  1.5 b/s  ETA 8.4min
  batch 1600/2240  mae=950.4s  1.5 b/s  ETA 7.2min
  batch 1700/2240  mae=940.8s  1.5 b/s  ETA 6.1min
  batch 1800/2240  mae=929.7s  1.5 b/s  ETA 5.0min
  bat

In [5]:
# CELL 8b — Reload NN checkpoint + regenerate nn_preds_dev (run after kernel restart)
import os
import numpy as np
import torch
import torch.nn as nn

# Use CPU to avoid CUDA arch mismatch after kernel restart
DEVICE   = 'cpu'
WORK_DIR = '/kaggle/working'
N_CONT   = 34
N_ZONES  = 266

FEATURE_NAMES = [
    'pair_mean','pair_median','pair_std','log_pair_count',
    'pair_hour_mean','pair_dow_mean','pair_month_mean',
    'pu_zone_mean','do_zone_mean','hour_pu_mean','hour_do_mean','boro_pair_mean',
    'dist_km','manhattan_km',
    'hour','dow','month','day','week',
    'hour_sin','hour_cos','dow_norm',
    'pickup_zone','dropoff_zone','pu_boro','do_boro',
    'same_zone','same_boro','is_airport_pu','is_airport_do',
    'is_holiday','is_christmas','is_nye','pax',
]

class ETANet(nn.Module):
    def __init__(self):
        super().__init__()
        self.pu_emb   = nn.Embedding(N_ZONES, 16, padding_idx=0)
        self.do_emb   = nn.Embedding(N_ZONES, 16, padding_idx=0)
        self.boro_emb = nn.Embedding(8, 4)
        self.hour_emb = nn.Embedding(24, 6)
        self.dow_emb  = nn.Embedding(7, 4)
        in_dim = 16+16+4+4+6+4 + N_CONT + 1
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 128),    nn.LayerNorm(128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64),     nn.GELU(),
            nn.Linear(64, 1),
        )
    def forward(self, cont, pu, do, pub, dob, h, dow, xgb_pred):
        x = torch.cat([
            self.pu_emb(pu), self.do_emb(do),
            self.boro_emb(pub), self.boro_emb(dob),
            self.hour_emb(h), self.dow_emb(dow),
            cont, xgb_pred.unsqueeze(1),
        ], dim=1)
        return self.net(x).squeeze(1)

# Load checkpoint onto CPU
nn_model = ETANet()  # no .to(DEVICE) needed, CPU is default
best_nn_state = torch.load(os.path.join(WORK_DIR, 'nn_checkpoint.pt'), map_location='cpu')
nn_model.load_state_dict(best_nn_state)
nn_model.eval()
print('NN checkpoint loaded on CPU ✓')

IDX    = {n:i for i,n in enumerate(FEATURE_NAMES)}
dv_pu  = X_dev[:, IDX['pickup_zone']].astype('int32').clip(1, 265)
dv_do  = X_dev[:, IDX['dropoff_zone']].astype('int32').clip(1, 265)
dv_pub = X_dev[:, IDX['pu_boro']].astype('int32').clip(0, 7)
dv_dob = X_dev[:, IDX['do_boro']].astype('int32').clip(0, 7)
dv_h   = X_dev[:, IDX['hour']].astype('int32').clip(0, 23)
dv_dow = X_dev[:, IDX['dow']].astype('int32').clip(0, 6)

print(f'Running inference on CPU ({len(X_dev):,} rows)...')
BATCH = 32768
nn_preds_dev = []
with torch.no_grad():
    for start in range(0, len(X_dev), BATCH):
        end   = min(start + BATCH, len(X_dev))
        xb    = torch.from_numpy(X_dev[start:end])
        pb    = torch.tensor(dv_pu[start:end],          dtype=torch.long)
        db    = torch.tensor(dv_do[start:end],          dtype=torch.long)
        pub_b = torch.tensor(dv_pub[start:end],         dtype=torch.long)
        dob_b = torch.tensor(dv_dob[start:end],         dtype=torch.long)
        hb    = torch.tensor(dv_h[start:end],           dtype=torch.long)
        dwb   = torch.tensor(dv_dow[start:end],         dtype=torch.long)
        xgb_b = torch.tensor(xgb_preds_dev[start:end], dtype=torch.float32)
        pred  = nn_model(xb, pb, db, pub_b, dob_b, hb, dwb, xgb_b)
        nn_preds_dev.append(pred.numpy())
        if (start // BATCH + 1) % 5 == 0:
            print(f'  {end:,}/{len(X_dev):,}', flush=True)

nn_preds_dev = np.concatenate(nn_preds_dev)
print(f'NN dev MAE: {np.mean(np.abs(nn_preds_dev - dev_y)):.2f}s')
print('Cell 8b done ✓ — ready for Cell 9')

NN checkpoint loaded on CPU ✓
Running inference on CPU (1,230,911 rows)...
  163,840/1,230,911
  327,680/1,230,911
  491,520/1,230,911
  655,360/1,230,911
  819,200/1,230,911
  983,040/1,230,911
  1,146,880/1,230,911
NN dev MAE: 263.63s
Cell 8b done ✓ — ready for Cell 9


In [6]:
# CELL 9 — Optimal ensemble blend
import os, gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from itertools import product

FEATURE_NAMES = [
    'pair_mean','pair_median','pair_std','log_pair_count',
    'pair_hour_mean','pair_dow_mean','pair_month_mean',
    'pu_zone_mean','do_zone_mean','hour_pu_mean','hour_do_mean','boro_pair_mean',
    'dist_km','manhattan_km',
    'hour','dow','month','day','week',
    'hour_sin','hour_cos','dow_norm',
    'pickup_zone','dropoff_zone','pu_boro','do_boro',
    'same_zone','same_boro','is_airport_pu','is_airport_do',
    'is_holiday','is_christmas','is_nye','pax',
]
DATA_DIR = '/kaggle/input/eta-data'
WORK_DIR = '/kaggle/working'

# Rebuild pair_dict from pair_mean column already in X_dev (col 0)
# Also rebuild dv_pu, dv_do from X_dev columns
IDX = {n:i for i,n in enumerate(FEATURE_NAMES)}
dv_pu  = X_dev[:, IDX['pickup_zone']].astype('int32').clip(1, 265)
dv_do  = X_dev[:, IDX['dropoff_zone']].astype('int32').clip(1, 265)
pair_mean_col = X_dev[:, IDX['pair_mean']]  # already computed per-row in engineer()
pair_preds = pair_mean_col.astype('float32')

# Rebuild lgb_preds_dev if lost
try:
    _ = cat_preds_dev
except NameError:
    import lightgbm as lgb
    lgb_model = lgb.Booster(model_file=os.path.join(WORK_DIR, 'lgb_model.txt'))
    cat_preds_dev = lgb_model.predict(X_dev).astype('float32')
    print(f'Reloaded LightGBM preds: MAE={np.mean(np.abs(cat_preds_dev - dev_y)):.1f}s')

# Rebuild xgb_preds_dev if lost
try:
    _ = xgb_preds_dev
except NameError:
    import xgboost as xgb
    xgb_model = xgb.Booster(); xgb_model.load_model(os.path.join(WORK_DIR, 'xgb_model.ubj'))
    xgb_preds_dev = xgb_model.predict(xgb.DMatrix(X_dev, feature_names=FEATURE_NAMES)).astype('float32')
    print(f'Reloaded XGBoost preds: MAE={np.mean(np.abs(xgb_preds_dev - dev_y)):.1f}s')

print('Individual MAEs on dev:')
print(f'  XGBoost:   {np.mean(np.abs(xgb_preds_dev - dev_y)):.2f}s')
print(f'  LightGBM:  {np.mean(np.abs(cat_preds_dev - dev_y)):.2f}s')
print(f'  NN:        {np.mean(np.abs(nn_preds_dev  - dev_y)):.2f}s')
print(f'  pair_mean: {np.mean(np.abs(pair_preds    - dev_y)):.2f}s')

best_mae = float('inf')
best_w   = (0.4, 0.3, 0.2, 0.1)
ws = np.arange(0, 1.01, 0.1)
for wx, wc, wn in product(ws, ws, ws):
    wl = round(1.0 - wx - wc - wn, 8)
    if wl < -1e-6 or wl > 1.0+1e-6: continue
    wl = max(0.0, wl)
    mae = float(np.mean(np.abs(wx*xgb_preds_dev + wc*cat_preds_dev + wn*nn_preds_dev + wl*pair_preds - dev_y)))
    if mae < best_mae:
        best_mae = mae; best_w = (float(wx), float(wc), float(wn), float(wl))

wx0, wc0, wn0, _ = best_w
for dx, dc, dn in product(np.arange(-0.1,0.11,0.02), np.arange(-0.1,0.11,0.02), np.arange(-0.1,0.11,0.02)):
    wx = round(wx0+dx,4); wc = round(wc0+dc,4); wn = round(wn0+dn,4)
    if wx<0 or wc<0 or wn<0: continue
    wl = round(1.0-wx-wc-wn, 8)
    if wl < 0: continue
    mae = float(np.mean(np.abs(wx*xgb_preds_dev + wc*cat_preds_dev + wn*nn_preds_dev + wl*pair_preds - dev_y)))
    if mae < best_mae:
        best_mae = mae; best_w = (wx, wc, wn, wl)

wx, wc, wn, wl = best_w
print(f'\nBest blend: xgb={wx:.3f}  lgb={wc:.3f}  nn={wn:.3f}  lookup={wl:.3f}')
print(f'Blend dev MAE: {best_mae:.2f}s')
blend_weights = best_w
print('Cell 9 done ✓')

Individual MAEs on dev:
  XGBoost:   262.15s
  LightGBM:  264.35s
  NN:        263.63s
  pair_mean: 301.22s

Best blend: xgb=0.180  lgb=0.320  nn=0.260  lookup=0.240
Blend dev MAE: 255.90s
Cell 9 done ✓


In [7]:
# CELL 10 — Save bundle
import pickle, os

BUNDLE_PATH = os.path.join(WORK_DIR, 'model_ensemble.pkl')

bundle = {
    'model_type':       'ensemble',
    'blend_weights':    blend_weights,
    'feature_names':    FEATURE_NAMES,
    'xgb_model':        xgb_model,
    'lgb_model':        lgb_model,
    'nn_state':         best_nn_state,
    'nn_config':        {'n_cont': N_CONT},
    'pair_agg':         stats['pair_agg'],
    'pu_mean':          stats['pu_mean'].to_dict(),
    'do_mean':          stats['do_mean'].to_dict(),
    'hour_pu_mean':     stats['hour_pu_mean'].to_dict(),
    'hour_do_mean':     stats['hour_do_mean'].to_dict(),
    'pair_hour_mean':   stats['pair_hour_mean'].to_dict(),
    'pair_dow_mean':    stats['pair_dow_mean'].to_dict(),
    'pair_month_mean':  stats['pair_month_mean'].to_dict(),
    'boro_pair_mean':   stats['boro_pair_mean'].to_dict(),
    'global_mean':      stats['global_mean'],
    'global_median':    stats['global_median'],
    'global_p75':       stats['global_p75'],
    'zone_meta':        zone_meta,
    'borough_cats':     dict(zip(stats['borough_cats'].index, stats['borough_cats'].values)),
    'dev_mae':          best_mae,
}

with open(BUNDLE_PATH, 'wb') as f:
    pickle.dump(bundle, f, protocol=4)

mb = os.path.getsize(BUNDLE_PATH)/1e6
print(f'Saved {BUNDLE_PATH}  ({mb:.0f} MB)')
print(f'Blend MAE: {best_mae:.2f}s')
print()
print('Files in /kaggle/working:')
for fn in sorted(os.listdir(WORK_DIR)):
    sz = os.path.getsize(os.path.join(WORK_DIR, fn))/1e6
    print(f'  {fn}  ({sz:.0f} MB)')
print('Cell 10 done ✓')

Saved /kaggle/working/model_ensemble.pkl  (46 MB)
Blend MAE: 255.90s

Files in /kaggle/working:
  .virtual_documents  (0 MB)
  lgb_model.txt  (24 MB)
  model_ensemble.pkl  (46 MB)
  nn_checkpoint.pt  (0 MB)
  state.db  (12835 MB)
  xgb_model.ubj  (5 MB)
Cell 10 done ✓


In [8]:
# CELL 11b — Re-save bundle WITHOUT xgb/lgb model objects (models loaded from .ubj/.txt at inference)
# This avoids pickle segfaults when loading on a different platform (macOS ARM vs Linux CUDA)
import pickle, os

BUNDLE_PATH     = '/kaggle/working/model_ensemble.pkl'
BUNDLE_SLIM_PATH = '/kaggle/working/model_ensemble_slim.pkl'

with open(BUNDLE_PATH, 'rb') as f:
    bundle = pickle.load(f)

# Remove the platform-specific model objects — they'll be loaded from files
bundle_slim = {k: v for k, v in bundle.items() if k not in ('xgb_model', 'lgb_model')}
bundle_slim['xgb_model_path'] = 'xgb_model.ubj'   # note: load separately
bundle_slim['lgb_model_path'] = 'lgb_model.txt'   # note: load separately

with open(BUNDLE_SLIM_PATH, 'wb') as f:
    pickle.dump(bundle_slim, f, protocol=4)

mb = os.path.getsize(BUNDLE_SLIM_PATH) / 1e6
print(f'Slim bundle saved: {BUNDLE_SLIM_PATH}  ({mb:.1f} MB)')
print(f'Keys: {list(bundle_slim.keys())}')
print('Download model_ensemble_slim.pkl from Output tab')

Slim bundle saved: /kaggle/working/model_ensemble_slim.pkl  (16.9 MB)
Keys: ['model_type', 'blend_weights', 'feature_names', 'nn_state', 'nn_config', 'pair_agg', 'pu_mean', 'do_mean', 'hour_pu_mean', 'hour_do_mean', 'pair_hour_mean', 'pair_dow_mean', 'pair_month_mean', 'boro_pair_mean', 'global_mean', 'global_median', 'global_p75', 'zone_meta', 'borough_cats', 'dev_mae', 'xgb_model_path', 'lgb_model_path']
Download model_ensemble_slim.pkl from Output tab


In [9]:
# CELL 11c — Run grade.py equivalent on Kaggle (avoids local platform issues)
import os, time
import numpy as np
import pandas as pd
import torch, torch.nn as nn
import xgboost as xgb
import lightgbm as lgb
from datetime import datetime

DATA_DIR = '/kaggle/input/eta-data'
WORK_DIR = '/kaggle/working'

FEATURE_NAMES = [
    'pair_mean','pair_median','pair_std','log_pair_count',
    'pair_hour_mean','pair_dow_mean','pair_month_mean',
    'pu_zone_mean','do_zone_mean','hour_pu_mean','hour_do_mean','boro_pair_mean',
    'dist_km','manhattan_km',
    'hour','dow','month','day','week',
    'hour_sin','hour_cos','dow_norm',
    'pickup_zone','dropoff_zone','pu_boro','do_boro',
    'same_zone','same_boro','is_airport_pu','is_airport_do',
    'is_holiday','is_christmas','is_nye','pax',
]

# Already in memory from Cell 7b — just verify
print(f'X_dev: {X_dev.shape}  dev_y: {dev_y.shape}')
print(f'xgb_preds_dev MAE: {np.mean(np.abs(xgb_preds_dev - dev_y)):.2f}s')
print(f'lgb_preds_dev MAE: {np.mean(np.abs(cat_preds_dev - dev_y)):.2f}s')
print(f'nn_preds_dev  MAE: {np.mean(np.abs(nn_preds_dev  - dev_y)):.2f}s')

wx, wc, wn, wl = blend_weights
IDX = {n:i for i,n in enumerate(FEATURE_NAMES)}
pair_preds = X_dev[:, IDX['pair_mean']].astype('float32')

blend = wx*xgb_preds_dev + wc*cat_preds_dev + wn*nn_preds_dev + wl*pair_preds
blend = np.maximum(blend, 30.0)
mae = float(np.mean(np.abs(blend - dev_y)))

print(f'\n=== FINAL ENSEMBLE DEV MAE: {mae:.2f}s ===')
print(f'Weights: xgb={wx:.3f}  lgb={wc:.3f}  nn={wn:.3f}  lookup={wl:.3f}')
print(f'vs LightGBM CPU baseline: 258.5s')
if mae < 258.5:
    print(f'IMPROVEMENT: {258.5-mae:.1f}s better ✓')
else:
    print('NOTE: does not beat 258.5s baseline')

X_dev: (1230911, 34)  dev_y: (1230911,)
xgb_preds_dev MAE: 262.15s
lgb_preds_dev MAE: 264.35s
nn_preds_dev  MAE: 263.63s

=== FINAL ENSEMBLE DEV MAE: 255.90s ===
Weights: xgb=0.180  lgb=0.320  nn=0.260  lookup=0.240
vs LightGBM CPU baseline: 258.5s
IMPROVEMENT: 2.6s better ✓
